# Querying AIStor Tables with Dremio

This notebook demonstrates how to:
1. Create Iceberg tables on AIStor using PyIceberg
2. Populate them with sample e-commerce data
3. Create an Iceberg View
4. Configure Dremio to connect to AIStor Tables
5. Query the tables from Dremio

**Architecture:**
```
Dremio (Query Engine)
  │
  ├─ Iceberg REST Catalog API (SigV4) ── minio:9000/_iceberg
  │   (table metadata, schema, snapshots)
  │
  └─ S3 API (data access) ───────────── minio:9000
      (Parquet data files)
```

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q "pyiceberg[s3fs]" boto3 requests pandas

In [ ]:
import os
import sys
import json
import requests
import pandas as pd
import pyarrow as pa
from datetime import datetime

# Add notebooks directory to path for sigv4 helper
sys.path.insert(0, os.path.dirname(os.path.abspath('sigv4.py')))
import sigv4

# Connection settings
HOST = os.environ.get("MINIO_HOST", "http://minio:9000")
ACCESS_KEY = "minioadmin"
SECRET_KEY = "minioadmin"
WAREHOUSE_NAME = "dremio-warehouse"

print(f"MinIO Host: {HOST}")
print(f"Warehouse:  {WAREHOUSE_NAME}")

## 2. Create Warehouse & Namespace

AIStor Tables uses the Iceberg REST Catalog API. We first create a **warehouse** (top-level container) and then a **namespace** to organize our tables.

In [ ]:
def make_request(method, path, body=""):
    """Make a SigV4-signed request to the AIStor Iceberg REST API."""
    url = f"{HOST}/_iceberg{path}"
    headers = {"Content-Type": "application/json"}
    signed = sigv4.sign(
        method, url, body, HOST, headers,
        access_key=ACCESS_KEY, secret_key=SECRET_KEY
    )
    resp = requests.request(
        method, url,
        headers=dict(signed.headers),
        data=body
    )
    return resp

# Create warehouse
resp = make_request("POST", "/v1/warehouses", json.dumps({
    "name": WAREHOUSE_NAME
}))
if resp.status_code in [200, 201]:
    print(f"Warehouse '{WAREHOUSE_NAME}' created")
elif resp.status_code == 409:
    print(f"Warehouse '{WAREHOUSE_NAME}' already exists")
else:
    print(f"Warehouse creation: {resp.status_code} - {resp.text}")

In [ ]:
# Create namespace
resp = make_request("POST", f"/v1/{WAREHOUSE_NAME}/namespaces", json.dumps({
    "namespace": ["sales"]
}))
if resp.status_code in [200, 201]:
    print("Namespace 'sales' created")
elif resp.status_code == 409:
    print("Namespace 'sales' already exists")
else:
    print(f"Namespace creation: {resp.status_code} - {resp.text}")

## 3. Initialize PyIceberg Catalog

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "aistor",
    type="rest",
    uri=f"{HOST}/_iceberg",
    warehouse=WAREHOUSE_NAME,
    **{
        "rest.sigv4-enabled": "true",
        "rest.signing-name": "s3tables",
        "rest.signing-region": "dummy",
        "s3.access-key-id": ACCESS_KEY,
        "s3.secret-access-key": SECRET_KEY,
        "s3.endpoint": HOST,
        "s3.path-style-access": "true",
    }
)

print("PyIceberg catalog connected")
print(f"Namespaces: {catalog.list_namespaces()}")

## 4. Create & Populate Tables

We'll create three related tables representing a simple e-commerce data model:
- **customers** - Customer profiles
- **products** - Product catalog
- **orders** - Sales transactions

### Customers Table

In [ ]:
customers_schema = pa.schema([
    pa.field("customer_id", pa.int64(), nullable=False),
    pa.field("name", pa.string(), nullable=False),
    pa.field("email", pa.string(), nullable=False),
    pa.field("city", pa.string()),
    pa.field("state", pa.string()),
    pa.field("country", pa.string()),
    pa.field("segment", pa.string()),
    pa.field("created_date", pa.string()),
])

try:
    catalog.drop_table("sales.customers")
except:
    pass

customers_table = catalog.create_table("sales.customers", schema=customers_schema)
print(f"Created table: sales.customers")

In [ ]:
customers_data = pa.table({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
    "name": [
        "Alice Johnson", "Bob Martinez", "Carol Williams", "David Lee", "Eva Brown",
        "Frank Chen", "Grace Kim", "Henry Kim", "Iris Patel", "Jack Thompson",
        "Karen Wilson", "Liam Davis", "Maria Rodriguez", "Nathan Park", "Olivia Taylor",
        "Paul Anderson", "Quinn Murphy", "Rachel Green", "Sam Wright", "Tina Foster"
    ],
    "email": [f"customer{i}@example.com" for i in range(1, 21)],
    "city": [
        "New York", "Los Angeles", "Chicago", "Houston", "Phoenix",
        "San Francisco", "Seattle", "Boston", "Denver", "Atlanta",
        "Miami", "Dallas", "Portland", "Austin", "Nashville",
        "Detroit", "Minneapolis", "San Diego", "Philadelphia", "Charlotte"
    ],
    "state": [
        "NY", "CA", "IL", "TX", "AZ", "CA", "WA", "MA", "CO", "GA",
        "FL", "TX", "OR", "TX", "TN", "MI", "MN", "CA", "PA", "NC"
    ],
    "country": ["US"] * 20,
    "segment": [
        "Enterprise", "SMB", "Enterprise", "Consumer", "SMB",
        "Enterprise", "Consumer", "SMB", "Enterprise", "Consumer",
        "SMB", "Enterprise", "Consumer", "SMB", "Enterprise",
        "Consumer", "SMB", "Enterprise", "Consumer", "SMB"
    ],
    "created_date": [
        "2023-01-15", "2023-02-20", "2023-03-10", "2023-04-05", "2023-05-12",
        "2023-06-18", "2023-07-22", "2023-08-30", "2023-09-14", "2023-10-25",
        "2023-11-08", "2023-12-01", "2024-01-15", "2024-02-20", "2024-03-10",
        "2024-04-05", "2024-05-12", "2024-06-18", "2024-07-22", "2024-08-30"
    ],
}, schema=customers_schema)

customers_table.append(customers_data)
print(f"Loaded {len(customers_data)} customers")

### Products Table

In [ ]:
products_schema = pa.schema([
    pa.field("product_id", pa.int64(), nullable=False),
    pa.field("product_name", pa.string(), nullable=False),
    pa.field("category", pa.string()),
    pa.field("unit_price", pa.float64()),
    pa.field("supplier", pa.string()),
])

try:
    catalog.drop_table("sales.products")
except:
    pass

products_table = catalog.create_table("sales.products", schema=products_schema)
print(f"Created table: sales.products")

In [ ]:
products_data = pa.table({
    "product_id": list(range(101, 116)),
    "product_name": [
        "Laptop Pro 15", "Wireless Mouse", "USB-C Hub", "Monitor 27in", "Mechanical Keyboard",
        "Webcam HD", "Headset Pro", "Docking Station", "External SSD 1TB", "4TB SSD",
        "Network Switch 24-port", "UPS Battery 1500VA", "Cable Kit", "Laptop Stand", "Desk Lamp LED"
    ],
    "category": [
        "Computers", "Accessories", "Accessories", "Displays", "Accessories",
        "Peripherals", "Audio", "Docking", "Storage", "Storage",
        "Networking", "Power", "Accessories", "Furniture", "Furniture"
    ],
    "unit_price": [
        1299.99, 29.99, 49.99, 449.99, 149.99,
        79.99, 199.99, 249.99, 109.99, 299.99,
        399.99, 189.99, 24.99, 59.99, 39.99
    ],
    "supplier": [
        "Dell", "Logitech", "Anker", "LG", "Keychron",
        "Logitech", "Jabra", "CalDigit", "Samsung", "Samsung",
        "Cisco", "APC", "Monoprice", "Rain Design", "BenQ"
    ],
}, schema=products_schema)

products_table.append(products_data)
print(f"Loaded {len(products_data)} products")

### Orders Table

In [ ]:
orders_schema = pa.schema([
    pa.field("order_id", pa.int64(), nullable=False),
    pa.field("customer_id", pa.int64(), nullable=False),
    pa.field("product_id", pa.int64(), nullable=False),
    pa.field("quantity", pa.int64()),
    pa.field("total_amount", pa.float64()),
    pa.field("order_date", pa.string()),
    pa.field("status", pa.string()),
    pa.field("region", pa.string()),
])

try:
    catalog.drop_table("sales.orders")
except:
    pass

orders_table = catalog.create_table("sales.orders", schema=orders_schema)
print(f"Created table: sales.orders")

In [ ]:
import random
random.seed(42)

n_orders = 50
statuses = ["Completed", "Completed", "Completed", "Shipped", "Processing", "Cancelled"]
regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West"]

order_ids = list(range(5001, 5001 + n_orders))
customer_ids = [random.randint(1, 20) for _ in range(n_orders)]
product_ids = [random.randint(101, 115) for _ in range(n_orders)]

# Look up prices
price_map = dict(zip(range(101, 116), [
    1299.99, 29.99, 49.99, 449.99, 149.99,
    79.99, 199.99, 249.99, 109.99, 299.99,
    399.99, 189.99, 24.99, 59.99, 39.99
]))

quantities = [random.randint(1, 10) for _ in range(n_orders)]
totals = [round(quantities[i] * price_map[product_ids[i]], 2) for i in range(n_orders)]

dates = []
for _ in range(n_orders):
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    dates.append(f"2024-{month:02d}-{day:02d}")

orders_data = pa.table({
    "order_id": order_ids,
    "customer_id": customer_ids,
    "product_id": product_ids,
    "quantity": quantities,
    "total_amount": totals,
    "order_date": dates,
    "status": [random.choice(statuses) for _ in range(n_orders)],
    "region": [random.choice(regions) for _ in range(n_orders)],
}, schema=orders_schema)

orders_table.append(orders_data)
print(f"Loaded {len(orders_data)} orders")

## 5. Create Iceberg View

Iceberg Views let you define virtual tables with SQL that live in the catalog. Dremio (and any other query engine) can query views just like tables.

In [ ]:
view_payload = json.dumps({
    "name": "revenue_by_region",
    "schema": {
        "type": "struct",
        "fields": [
            {"id": 1, "name": "region", "type": "string", "required": False},
            {"id": 2, "name": "total_revenue", "type": "double", "required": False},
            {"id": 3, "name": "order_count", "type": "long", "required": False}
        ]
    },
    "view-version": {
        "version-id": 1,
        "schema-id": 0,
        "timestamp-ms": int(datetime.now().timestamp() * 1000),
        "summary": {"engine-name": "PyIceberg"},
        "representations": [
            {
                "type": "sql",
                "sql": "SELECT region, SUM(total_amount) as total_revenue, COUNT(*) as order_count FROM sales.orders GROUP BY region ORDER BY total_revenue DESC",
                "dialect": "spark"
            }
        ],
        "default-namespace": ["sales"]
    }
})

resp = make_request("POST", f"/v1/{WAREHOUSE_NAME}/namespaces/sales/views", view_payload)
if resp.status_code in [200, 201]:
    print("View 'revenue_by_region' created")
elif resp.status_code == 409:
    print("View 'revenue_by_region' already exists")
else:
    print(f"View creation: {resp.status_code} - {resp.text}")

## 6. Verify Data via PyIceberg

In [ ]:
print("=== Tables in 'sales' namespace ===")
for table_id in catalog.list_tables("sales"):
    table = catalog.load_table(table_id)
    scan = table.scan()
    df = scan.to_pandas()
    print(f"  {table_id[1]}: {len(df)} rows")

print("\n=== Sample: customers ===")
print(catalog.load_table("sales.customers").scan().to_pandas().head())

print("\n=== Sample: products ===")
print(catalog.load_table("sales.products").scan().to_pandas().head())

print("\n=== Sample: orders ===")
print(catalog.load_table("sales.orders").scan().to_pandas().head())

---

## 7. Configure Dremio Source

### Step 1: Create your Dremio admin account

Before running the cells below, open **http://localhost:9047** in your browser and create your admin account.

Then update the credentials in the next cell.

In [ ]:
# Update these with the admin account you created in Dremio
# (matches what you set during first-time setup at http://localhost:9047)
DREMIO_HOST = "http://dremio:9047"
DREMIO_USER = os.environ.get("DREMIO_USER", "admin")
DREMIO_PASS = os.environ.get("DREMIO_PASS", "")

if not DREMIO_PASS:
    raise ValueError("Set DREMIO_PASS to your Dremio admin password before running this cell.\n"
                     "  In Jupyter: kernel > restart, then set os.environ['DREMIO_PASS'] = 'yourpassword'\n"
                     "  Or set DREMIO_PASS environment variable before starting Jupyter.")

### Step 2: Authenticate with Dremio API

In [ ]:
# Get Dremio auth token
auth_resp = requests.post(
    f"{DREMIO_HOST}/apiv2/login",
    json={"userName": DREMIO_USER, "password": DREMIO_PASS}
)
auth_resp.raise_for_status()
DREMIO_TOKEN = auth_resp.json()["token"]
DREMIO_HEADERS = {
    "Authorization": f"_dremio{DREMIO_TOKEN}",
    "Content-Type": "application/json"
}
print("Authenticated with Dremio")

### Step 3: Create RESTCATALOG Source

This connects Dremio to AIStor's built-in Iceberg REST Catalog.

**Key configuration details:**
- `restEndpointUri` - The Iceberg REST API endpoint (includes `http://`)
- `fs.s3a.endpoint` - The S3 data endpoint (**must NOT include `http://`** - just `host:port`)
- `fs.s3a.connection.ssl.enabled` - Must be `false` for non-TLS connections
- `dremio.s3.compat` - Must be `true` for S3-compatible storage (MinIO)
- `rest.sigv4-enabled` - AIStor Tables uses SigV4 authentication
- `rest.access-key-id` / `rest.secret-access-key` - Go in **propertyList** (not secretPropertyList)

In [ ]:
source_config = {
    "entityType": "source",
    "name": "aistor_tables",
    "type": "RESTCATALOG",
    "config": {
        "restEndpointUri": "http://minio:9000/_iceberg",
        "isUsingVendedCredentials": False,
        "enableAsync": True,
        "isCachingEnabled": True,
        "maxCacheSpacePct": 100,
        "isRecursiveAllowedNamespaces": True,
        "propertyList": [
            {"name": "warehouse", "value": WAREHOUSE_NAME},
            {"name": "rest.sigv4-enabled", "value": "true"},
            {"name": "rest.signing-name", "value": "s3tables"},
            {"name": "rest.signing-region", "value": "dummy"},
            {"name": "rest.access-key-id", "value": ACCESS_KEY},
            {"name": "rest.secret-access-key", "value": SECRET_KEY},
            {"name": "fs.s3a.endpoint", "value": "minio:9000"},
            {"name": "fs.s3a.path.style.access", "value": "true"},
            {"name": "fs.s3a.connection.ssl.enabled", "value": "false"},
            {"name": "fs.s3a.aws.credentials.provider", "value": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"},
            {"name": "dremio.s3.compat", "value": "true"}
        ],
        "secretPropertyList": [
            {"name": "fs.s3a.access.key", "value": ACCESS_KEY},
            {"name": "fs.s3a.secret.key", "value": SECRET_KEY}
        ]
    }
}

# Create or update source
resp = requests.post(
    f"{DREMIO_HOST}/api/v3/catalog",
    headers=DREMIO_HEADERS,
    json=source_config
)

if resp.status_code in [200, 201]:
    source_data = resp.json()
    print(f"Source 'aistor_tables' created")
    print(f"Source ID: {source_data.get('id', 'N/A')}")
elif resp.status_code == 409:
    # Source already exists - update it
    print("Source 'aistor_tables' already exists, updating...")
    existing = requests.get(
        f"{DREMIO_HOST}/api/v3/catalog/by-path/aistor_tables",
        headers=DREMIO_HEADERS
    )
    if existing.status_code == 200:
        existing_data = existing.json()
        source_config["tag"] = existing_data.get("tag")
        update_resp = requests.put(
            f"{DREMIO_HOST}/api/v3/catalog/{existing_data['id']}",
            headers=DREMIO_HEADERS,
            json=source_config
        )
        if update_resp.status_code == 200:
            print("Source updated successfully")
        else:
            print(f"Source update failed: {update_resp.status_code} - {update_resp.text}")
else:
    print(f"Source creation failed: {resp.status_code}")
    print(resp.text)

In [ ]:
# Verify source status
import time
time.sleep(5)  # Give Dremio a moment to connect

resp = requests.get(
    f"{DREMIO_HOST}/api/v3/catalog/by-path/aistor_tables",
    headers=DREMIO_HEADERS
)
if resp.status_code == 200:
    source_info = resp.json()
    status = source_info.get("status", {}).get("state", "unknown")
    print(f"Source status: {status}")
else:
    print(f"Could not check source: {resp.status_code}")

## 8. Query from Dremio

Now we can run SQL queries through Dremio against the AIStor Tables data.

In [ ]:
import time

def dremio_query(sql, timeout=120):
    """Submit a SQL query to Dremio and return results as a DataFrame."""
    # Submit query
    resp = requests.post(
        f"{DREMIO_HOST}/api/v3/sql",
        headers=DREMIO_HEADERS,
        json={"sql": sql}
    )
    resp.raise_for_status()
    job_id = resp.json()["id"]
    
    # Poll for completion
    start = time.time()
    while time.time() - start < timeout:
        status_resp = requests.get(
            f"{DREMIO_HOST}/api/v3/job/{job_id}",
            headers=DREMIO_HEADERS
        )
        status_resp.raise_for_status()
        job_info = status_resp.json()
        state = job_info["jobState"]
        
        if state == "COMPLETED":
            # Fetch results
            results_resp = requests.get(
                f"{DREMIO_HOST}/api/v3/job/{job_id}/results?offset=0&limit=500",
                headers=DREMIO_HEADERS
            )
            results_resp.raise_for_status()
            data = results_resp.json()
            columns = [col["name"] for col in data["schema"]]
            return pd.DataFrame(data["rows"], columns=columns)
        elif state == "FAILED":
            raise Exception(f"Query failed: {job_info.get('errorMessage', 'Unknown error')}")
        elif state == "CANCELLED":
            raise Exception("Query was cancelled")
        
        time.sleep(2)
    
    raise TimeoutError(f"Query did not complete within {timeout} seconds")

### Query: All Customers

In [ ]:
df = dremio_query("SELECT * FROM aistor_tables.sales.customers ORDER BY customer_id LIMIT 10")
print(f"Returned {len(df)} rows")
df

### Query: Product Catalog

In [ ]:
df = dremio_query("SELECT * FROM aistor_tables.sales.products ORDER BY unit_price DESC")
print(f"Returned {len(df)} rows")
df

### Query: Top Orders by Revenue

In [ ]:
df = dremio_query("""
    SELECT 
        c.name AS customer,
        p.product_name,
        o.quantity,
        o.total_amount,
        o.order_date,
        o.region
    FROM aistor_tables.sales.orders o
    JOIN aistor_tables.sales.customers c ON o.customer_id = c.customer_id
    JOIN aistor_tables.sales.products p ON o.product_id = p.product_id
    ORDER BY o.total_amount DESC
    LIMIT 15
""")
print(f"Top 15 orders by revenue:")
df

### Query: Revenue by Region

In [ ]:
df = dremio_query("""
    SELECT 
        region,
        COUNT(*) AS order_count,
        SUM(total_amount) AS total_revenue,
        ROUND(AVG(total_amount), 2) AS avg_order_value
    FROM aistor_tables.sales.orders
    WHERE status = 'Completed'
    GROUP BY region
    ORDER BY total_revenue DESC
""")
print("Revenue by region (completed orders):")
df

### Query: Customer Segment Analysis

In [ ]:
df = dremio_query("""
    SELECT 
        c.segment,
        COUNT(DISTINCT c.customer_id) AS customers,
        COUNT(o.order_id) AS orders,
        ROUND(SUM(o.total_amount), 2) AS total_revenue,
        ROUND(AVG(o.total_amount), 2) AS avg_order_value
    FROM aistor_tables.sales.customers c
    LEFT JOIN aistor_tables.sales.orders o ON c.customer_id = o.customer_id
    GROUP BY c.segment
    ORDER BY total_revenue DESC
""")
print("Customer segment analysis:")
df

### Query: Top-Selling Products

In [ ]:
df = dremio_query("""
    SELECT 
        p.product_name,
        p.category,
        p.supplier,
        COUNT(o.order_id) AS times_ordered,
        SUM(o.quantity) AS total_units,
        ROUND(SUM(o.total_amount), 2) AS total_revenue
    FROM aistor_tables.sales.products p
    JOIN aistor_tables.sales.orders o ON p.product_id = o.product_id
    GROUP BY p.product_name, p.category, p.supplier
    ORDER BY total_revenue DESC
""")
print("Top-selling products:")
df

---

## 9. Example SQL Queries for Dremio UI

You can also run these queries directly in the Dremio SQL editor at **http://localhost:9047**:

```sql
-- All customers
SELECT * FROM aistor_tables.sales.customers;

-- Product catalog
SELECT * FROM aistor_tables.sales.products ORDER BY unit_price DESC;

-- Top orders with customer and product details
SELECT 
    c.name AS customer,
    p.product_name,
    o.quantity,
    o.total_amount,
    o.order_date,
    o.region
FROM aistor_tables.sales.orders o
JOIN aistor_tables.sales.customers c ON o.customer_id = c.customer_id
JOIN aistor_tables.sales.products p ON o.product_id = p.product_id
ORDER BY o.total_amount DESC
LIMIT 20;

-- Revenue by region
SELECT 
    region,
    COUNT(*) AS order_count,
    SUM(total_amount) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS avg_order_value
FROM aistor_tables.sales.orders
WHERE status = 'Completed'
GROUP BY region
ORDER BY total_revenue DESC;

-- Monthly sales trend
SELECT 
    SUBSTRING(order_date, 1, 7) AS month,
    COUNT(*) AS orders,
    ROUND(SUM(total_amount), 2) AS revenue
FROM aistor_tables.sales.orders
GROUP BY SUBSTRING(order_date, 1, 7)
ORDER BY month;
```

## Summary

This notebook demonstrated the full workflow for using Dremio with AIStor Tables:

1. **Created Iceberg tables** on AIStor using PyIceberg with SigV4 authentication
2. **Populated tables** with sample e-commerce data (customers, products, orders)
3. **Created an Iceberg View** for pre-defined analytics
4. **Configured Dremio** to connect via the RESTCATALOG source type
5. **Ran SQL queries** through Dremio including multi-table joins

### Key Configuration Notes

| Property | Value | Notes |
|----------|-------|-------|
| `restEndpointUri` | `http://minio:9000/_iceberg` | Includes `http://` |
| `fs.s3a.endpoint` | `minio:9000` | **NO** `http://` prefix |
| `fs.s3a.connection.ssl.enabled` | `false` | Required for non-TLS |
| `dremio.s3.compat` | `true` | Required for MinIO/S3-compatible storage |
| `rest.sigv4-enabled` | `true` | AIStor uses SigV4 auth |
| `rest.signing-name` | `s3tables` | Service name for SigV4 |
| `rest.access-key-id` | (in propertyList) | **Not** in secretPropertyList |
| `fs.s3a.path.style.access` | `true` | Required for MinIO |

### Dremio API Auth

Dremio uses `_dremio{TOKEN}` format for the Authorization header (not `Bearer`).